# Loading of sessions

## Imports and session loading

In [ ]:
PROJECT_DIR = "./salient_kpms_1_ld7_jn24"  # Using a new directory for the full analysis
MODEL_NAME = 'final_model_3.00e+10_idx0'

In [ ]:
# --- IMPORTS & SETUP ---

# Enable autoreload (optional but good practice)
# %load_ext autoreload
# %autoreload 2

# Standard library & path setup
import sys
from pathlib import Path

# Data handling and analysis
import pandas as pd
import numpy as np
import pickle
import re

# Visualization
import matplotlib.pyplot as plt
import pprint

# Modeling pipeline
import keypoint_moseq as kpms
import jax

# --- 1. SETUP PROJECT PATH ---
print("Setting up project path...")
project_root = Path.cwd()
if not (project_root / "fiberphotometry").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))
print(f"Project root added to path: {project_root}")

In [ ]:
from fiberphotometry.kpms.trial import DFStore, Trial

In [ ]:
import pickle
import pandas as pd
import numpy as np

# === Load state ===
load_filename = '/home/fsp585/Desktop/FiberphotometryCode/tests/salient_kappa_3e10.pkl'
print(f"--- 🚀 Loading complete analysis state from '{load_filename}'... ---")

with open(load_filename, 'rb') as f:
    loaded_state = pickle.load(f)

# Unpack (tolerant to naming)
objs = loaded_state.get('sessions', loaded_state.get('trials'))
if objs is None:
    raise KeyError("Pickle missing both 'sessions' and 'trials'.")

PROJECT_DIR_FULL = loaded_state.get('project_dir_full')
KAPPA = loaded_state.get('kappa')
video_frame_indexes = loaded_state['video_frame_indexes']
original_lengths = loaded_state['original_lengths']
final_model_names = loaded_state.get('final_model_names', loaded_state.get('model_names', []))

print(f"--- ✅ Base state loaded. Objects: {len(objs)}. Candidate models: {len(final_model_names)} ---")

# ---- materialize a single-model view into .all_results (unchanged logic) ----
def _materialize_single_model_view(objects, model_name):
    updated, missing = 0, []
    for s in objects:
        store = getattr(s, 'kpms_results_by_model', None)
        if not store or model_name not in store:
            missing.append(getattr(s, 'trial_id', getattr(s, 'name', '?')))
            if hasattr(s, 'all_results'):
                delattr(s, 'all_results')
            continue

        res = store[model_name]
        # create local alias if only 'syllables' exists
        if 'syllable' not in res and 'syllables' in res:
            res = {**res, 'syllable': res['syllables']}

        if hasattr(s, 'all_results'):
            delattr(s, 'all_results')
        setattr(s, 'all_results', res)
        updated += 1
    return updated, missing

# Ensure MODEL_NAME is set
if 'MODEL_NAME' not in globals():
    raise NameError("Please set MODEL_NAME before loading (e.g., MODEL_NAME = final_model_names[0]).")

updated, missing = _materialize_single_model_view(objs, MODEL_NAME)
print(f"✅ Materialized single-model view: {updated} objects now have 'all_results' for '{MODEL_NAME}'.")
if missing:
    print(f"⚠️ {len(missing)} objects lacked results for '{MODEL_NAME}': {missing[:6]}{' ...' if len(missing)>6 else ''}")

any_ok = any(
    hasattr(s, 'all_results') and isinstance(s.all_results, dict) and
    (('syllable' in s.all_results) or ('syllables' in s.all_results))
    for s in objs
)
print("🔎 Post-load check:", "OK ✅" if any_ok else "Issue ❌ — no objects expose syllables via all_results")

## Syllable syncing

In [ ]:
sessions = loaded_state['sessions']

In [ ]:
sessions[0].dfs.keys()

In [ ]:
# --- Pre-clean: drop unused columns like 'ExtraVar1' before syncing ---
drop_cols = {"ExtraVar1", "Genotype", "MouseNumber"}  # add more if needed

dropped = {}
for i, s in enumerate(sessions):
    sid = getattr(s, "trial_id", f"trial_{i+1}")
    dropped[sid] = []
    for name in ("signal", "tracking"):
        if name in s.dfs:
            cols = [c for c in s.dfs[name].columns if c in drop_cols]
            if cols:
                s.dfs[name].drop(columns=cols, inplace=True, errors="ignore")
                dropped[sid].extend([f"{name}:{c}" for c in cols])

# (optional) quick report
summary = {k: v for k, v in dropped.items() if v}
print("Dropped columns:", summary if summary else "(none)")


In [ ]:
# === BIG synced DataFrame (DLC timebase) for Trial objects ===
# Uses: s.dfs['signal'], s.dfs['tracking'], s.all_results (syllable etc.)
# Master grid: DLC @60 Hz (length = len(s.all_results['syllable']))
# Tracking: dense nearest (±½ frame), Signal: sparse (true samples only)

print("--- 🧠 Building DLC-synced DataFrames (handles 2D results like latent_state) ---")

objs = sessions  # your trials list
dlc_fps = 60.0
tol = 0.5 / dlc_fps  # ± half-frame

def _to_num(s):
    return pd.to_numeric(s, errors="coerce")

def _median_offset_to_grid(trk_times, grid_times):
    left  = pd.DataFrame({"trk": _to_num(trk_times)}).dropna().sort_values("trk")
    right = pd.DataFrame({"time": pd.Series(grid_times, dtype="float64")}).sort_values("time")
    matched = pd.merge_asof(left, right, left_on="trk", right_on="time",
                            direction="nearest", tolerance=tol)
    deltas = (matched["trk"] - matched["time"]).dropna()
    if deltas.empty:
        raise ValueError("Could not match tracking to DLC grid within tolerance.")
    return float(deltas.median())

def _expand_results_to_df(res, index):
    """
    Res (dict of arrays). Expand 2D arrays into multiple columns.
    Special-cases: centroid -> centroid_x, centroid_y when shape[1]==2.
    """
    cols = {}
    N = len(index)
    for k, v in res.items():
        arr = np.asarray(v)
        if arr.shape[0] != N:
            raise ValueError(f"all_results['{k}'] length {arr.shape[0]} != grid length {N}")
        if arr.ndim == 1:
            cols[k] = arr
        elif arr.ndim == 2:
            D = arr.shape[1]
            if k == "centroid" and D == 2:
                cols["centroid_x"] = arr[:, 0]
                cols["centroid_y"] = arr[:, 1]
            else:
                for j in range(D):
                    cols[f"{k}_{j}"] = arr[:, j]
        else:
            raise ValueError(f"Unsupported ndim for '{k}': {arr.ndim}")
    return pd.DataFrame(cols, index=index)

updated = 0
for i, s in enumerate(objs):
    sid = getattr(s, "trial_id", f"trial_{i+1}")
    print(f"\n→ {i+1}/{len(objs)}: {sid}")

    # sanity checks
    if "signal" not in s.dfs or "tracking" not in s.dfs:
        print("  ❌ missing 'signal' or 'tracking'; skipping")
        continue
    if not hasattr(s, "all_results") or not isinstance(s.all_results, dict):
        print("  ❌ missing 'all_results'; skipping")
        continue

    sig = s.dfs["signal"].copy()
    trk = s.dfs["tracking"].copy()
    res = s.all_results

    # master DLC grid length
    if "syllable" in res:
        N = len(res["syllable"])
    else:
        # fallback: use the first entry length
        first_key = next(iter(res))
        N = len(res[first_key])
    grid = pd.DataFrame({"time": np.arange(N, dtype="float64") / dlc_fps})
    grid_index = pd.Index(grid["time"].values, name="time")

    # (A) syllables/results directly on DLC grid (expand 2D arrays)
    syl_df = _expand_results_to_df(res, grid_index)

    # (B) ensure tracking has synced time relative to photometry zero
    if "SecFromZero_sync" not in trk.columns:
        if "BonsaiTrackingTimestamp" in trk.columns and "BonsaiFlyTimestamp" in sig.columns:
            t0 = _to_num(sig["BonsaiFlyTimestamp"].iloc[0])
            trk["SecFromZero_sync"] = _to_num(trk["BonsaiTrackingTimestamp"]) - t0
        else:
            raise ValueError(f"{sid}: need 'SecFromZero_sync' (or raw timestamp columns) in tracking")

    # estimate global offset Δ (tracking ≈ DLC + Δ)
    delta = _median_offset_to_grid(trk["SecFromZero_sync"], grid_index.values)

    # (C) tracking → DLC grid (dense)
    trk_sorted = (
        trk.assign(_time_dlc=_to_num(trk["SecFromZero_sync"]) - delta)
           .reset_index(drop=False)
           .rename(columns={"index": "tracking_index"})
           .sort_values("_time_dlc")
    )
    trk_on = pd.merge_asof(
        left=grid, right=trk_sorted,
        left_on="time", right_on="_time_dlc",
        direction="nearest", tolerance=tol
    ).set_index("time")
    if "_time_dlc" in trk_on.columns:
        trk_on.drop(columns=["_time_dlc"], inplace=True)

    # (D) signal → DLC grid (sparse: snap each true sample once)
    if "SecFromZero_signal" not in sig.columns:
        raise ValueError(f"{sid}: signal missing 'SecFromZero_signal'")
    sig_sorted = (
        sig.assign(_time_dlc=_to_num(sig["SecFromZero_signal"]) - delta)
           .reset_index(drop=False)
           .rename(columns={"index": "signal_index"})
           .sort_values("_time_dlc")
    )
    sig_to_grid = pd.merge_asof(
        left=sig_sorted, right=grid,
        left_on="_time_dlc", right_on="time",
        direction="nearest", tolerance=tol
    )
    sig_on = (
        sig_to_grid.dropna(subset=["time"])
                   .set_index("time")
    )
    # keep only the first hit per tick; then reindex sparsely
    sig_on = sig_on[~sig_on.index.duplicated(keep="first")]
    if "_time_dlc" in sig_on.columns:
        sig_on.drop(columns=["_time_dlc"], inplace=True)
    sig_on = sig_on.reindex(grid_index)

    # collision checks (no prefixes)
    def _collide(a, b): return sorted(set(a.columns) & set(b.columns))
    c1 = _collide(syl_df, trk_on)
    c2 = _collide(syl_df, sig_on)
    c3 = _collide(trk_on, sig_on)
    if c1 or c2 or c3:
        msgs = []
        if c1: msgs.append("syllables↔tracking: " + ", ".join(c1))
        if c2: msgs.append("syllables↔signal: " + ", ".join(c2))
        if c3: msgs.append("tracking↔signal: " + ", ".join(c3))
        raise ValueError(f"{sid}: column collision(s) without prefixes: " + " | ".join(msgs))

    # final join: results + tracking (dense) + signal (sparse)
    big = syl_df.join(trk_on, how="left").join(sig_on, how="left")
    big.index.name = "time"
    big.insert(0, "frame", np.arange(N, dtype=np.int64))  # convenience

    s.dfs["aligned_60hz"] = big
    updated += 1
    print(f"  ✅ synced: {len(big)} rows | cols={len(big.columns)} → stored as 'aligned_60hz'")

print(f"\n--- ✅ Done. Built DLC-synced DataFrames for {updated}/{len(objs)} trials. ---")

# Peek
if updated:
    print("\nFirst trial snapshot:")
    print(objs[0].dfs["aligned_60hz"].info())
    print(objs[0].dfs["aligned_60hz"].head(3))


In [ ]:
objs[0].dfs["aligned_60hz"].info()

## Model loading and more sanity checks

In [ ]:
# # define your ten model names
# model_names = [f'final_model_3.00e+10_idx{i}' for i in range(3)]

# # compute EML scores and their standard errors
# eml_scores, eml_std_errs = kpms.expected_marginal_likelihoods(PROJECT_DIR, model_names)

# # pick the best model
# best_model_name = model_names[np.argmax(eml_scores)]
# print(f"Best model: {best_model_name}")

# # visualize the EML scores across the ensemble
# kpms.plot_eml_scores(eml_scores, eml_std_errs, model_names)


In [ ]:
import keypoint_moseq as kpms
model, data, metadata, current_iter = kpms.load_checkpoint(PROJECT_DIR, MODEL_NAME)
results = kpms.extract_results(model, metadata, PROJECT_DIR, MODEL_NAME, save_results=True)

In [ ]:
# model_names = [f'final_model_3.00e+10_idx{i}' for i in range(3)]

# for model_name in model_names:
#     # load the specific checkpoint
#     model, data, metadata, current_iter = kpms.load_checkpoint(PROJECT_DIR, model_name)
    
#     # extract results for this model
#     results = kpms.extract_results(model, metadata, PROJECT_DIR, model_name, save_results=True)


In [ ]:
config = kpms.load_config(PROJECT_DIR)

In [ ]:
import os
import h5py
import numpy as np
import jax.numpy as jnp
from jax_moseq.utils import unbatch
from jax_moseq.models.keypoint_slds import estimate_coordinates

# compute the estimated coordinates
Y_est = estimate_coordinates(
    jnp.array(model['states']['x']),
    jnp.array(model['states']['v']),
    jnp.array(model['states']['h']),
    jnp.array(model['params']['Cd'])
)

# generate a dictionary with reconstructed coordinates for each recording
coordinates_est = unbatch(Y_est, *metadata)

In [ ]:
config['bodyparts'] = config['use_bodyparts']

In [ ]:
centroids, headings = kpms.get_centroids_headings(coordinates_est, **config)

## Group assignment and syllable labeling

In [ ]:
import re

# metadata_names is the list/array of all result names (e.g., metadata[0])
metadata_names = metadata[0]

for sess in sessions:
    sess.result_names = []
    h5_path = sess.kpms_h5_path
    for name in metadata_names:
        # strip "_part#" only for matching
        base = re.sub(r'_part\d+$', '', name)
        if base in h5_path and name not in sess.result_names:
            # append the full original name (with part suffix)
            sess.result_names.append(name)

In [ ]:
import pandas as pd

# build a list of dicts
rows = []
for sess in sessions:
    for name in sess.result_names:
        rows.append({
            "name": name,
            "group": sess.genotype
        })

# make a DataFrame
index_df = pd.DataFrame(rows)

# (optionally) save it back out so your interactive widget picks it up
index_df.to_csv(os.path.join(PROJECT_DIR, "index.csv"), index=False)

# inspect
print(index_df.head())


In [ ]:
moseq_df = kpms.compute_moseq_df(PROJECT_DIR, MODEL_NAME, smooth_heading=True)

In [ ]:
moseq_df

In [ ]:
stats_df = kpms.compute_stats_df(
    PROJECT_DIR,
    MODEL_NAME,
    moseq_df,
    min_frequency=0.005,  # threshold frequency for including a syllable in the dataframe
    groupby=["group", "name"],  # column(s) to group the dataframe by
    fps=60,
)  # frame rate of the video from which keypoints were inferred

stats_df

In [ ]:
sessions[0].kpms_h5_path

## Built-in analyzis

In [ ]:
kpms.plot_syll_stats_with_sem(
    stats_df,
    PROJECT_DIR,
    MODEL_NAME,
    plot_sig=True,  # whether to mark statistical significance with a star
    thresh=0.05,  # significance threshold
    stat="frequency",  # statistic to be plotted (e.g. 'duration' or 'velocity_px_s_mean')
    order="stat",  # order syllables by overall frequency ("stat") or degree of difference ("diff")
    ctrl_group="a",  # name of the control group for statistical testing
    exp_group="b",  # name of the experimental group for statistical testing
    figsize=(8, 4),  # figure size
    groups=stats_df["group"].unique(),  # groups to be plotted
);


In [ ]:
normalize = "bigram"  # normalization method ("bigram", "rows" or "columns")

trans_mats, usages, groups, syll_include = kpms.generate_transition_matrices(
    PROJECT_DIR,
    MODEL_NAME,
    normalize=normalize,
    min_frequency=0.010,  # minimum syllable frequency to include
)

kpms.visualize_transition_bigram(
    PROJECT_DIR,
    MODEL_NAME,
    groups,
    trans_mats,
    syll_include,
    normalize=normalize,
    show_syllable_names=True,  # label syllables by index (False) or index and name (True)
)


In [ ]:

kpms.plot_transition_graph_group(
    PROJECT_DIR,
    MODEL_NAME,
    groups,
    trans_mats,
    usages,
    syll_include,
    layout="circular",  # transition graph layout ("circular" or "spring")
    show_syllable_names=False,  # label syllables by index (False) or index and name (True)
)


In [ ]:
kpms.plot_transition_graph_group(
    PROJECT_DIR,
    MODEL_NAME,
    groups,
    trans_mats,
    usages,
    syll_include,
    layout="circular",  # transition graph layout ("circular" or "spring")
    show_syllable_names=True,  # label syllables by index (False) or index and name (True)
)


In [ ]:
kpms.plot_transition_graph_difference(
    PROJECT_DIR, MODEL_NAME, groups, trans_mats, usages, syll_include, layout="circular", show_syllable_names=True
)  # transition graph layout ("circular" or "spring")

## PETHs

In [ ]:
sessions[0].dfs

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import sem
from scipy.signal import savgol_filter
import os

In [ ]:
# --- where to save (same as before) ---
fig_base_dir = os.path.join("syllable_figs", MODEL_NAME)
os.makedirs(fig_base_dir, exist_ok=True)

# --- params (60 Hz DLC grid) ---
SAMPLING_RATE = 60
WINDOW_SECONDS = 10
SMOOTHING_WINDOW = 25     # kept; code makes it odd & <= length
SMOOTHING_POLY_ORDER = 3
CONFIDENCE_LEVEL = 1.96
WINDOW_SAMPLES = int(WINDOW_SECONDS * SAMPLING_RATE)
TOTAL_SAMPLES = 2 * WINDOW_SAMPLES
MIN_TRACES_PER_SYLLABLE = 10  # set 0 to include all

# prefer the most processed/z-scored columns, fall back gracefully
PREFERRED_SIGNAL_PAIRS = [
    ("VS_Scaled_ZdFF_detrend", "DS_Scaled_ZdFF_detrend"),
    ("VS_Scaled_dFF_detrend",  "DS_Scaled_dFF_detrend"),
    ("VS_Scaled_dFF",          "DS_Scaled_dFF"),
    ("VS",                     "DS"),
]

def pick_signal_cols(df):
    cols = set(df.columns)
    for a, b in PREFERRED_SIGNAL_PAIRS:
        if a in cols and b in cols:
            return [a, b]
    raise KeyError(f"No suitable VS/DS signal columns found in df columns:\n{sorted(cols)}")

def _interp_then_smooth(x, win, poly):
    x = np.asarray(x, dtype=float)
    n = x.size
    if np.isnan(x).any():
        idx = np.arange(n)
        good = ~np.isnan(x)
        if good.any():
            # linear interp; hold edges
            x[~good] = np.interp(idx[~good], idx[good], x[good])
            if np.isnan(x[0]): x[0] = x[good][0]
            if np.isnan(x[-1]): x[-1] = x[good][-1]
        else:
            return np.zeros_like(x)
    # make window odd and <= n
    w = min(win, n - (n % 2 == 0))
    if w < 5: 
        return x
    if w % 2 == 0:
        w -= 1
    if w <= poly:
        poly = max(1, min(poly, w - 2))
    return savgol_filter(x, w, poly)

def get_onset_positions(syl_series):
    """boolean change detector on the DLC-indexed syllable series -> onset positions (int)."""
    v = syl_series.to_numpy()
    # onset when current is finite and different from previous (or prev nan)
    change = np.empty_like(v, dtype=bool)
    change[0] = np.isfinite(v[0])
    prev = v[:-1]
    cur = v[1:]
    change[1:] = np.isfinite(cur) & (np.isnan(prev) | (cur != prev))
    return np.where(change)[0]

def process_trial(trial, signal_cols):
    """Extract baseline-centered, smoothed traces around onsets for one Trial."""
    if "aligned_60hz" not in trial.dfs:
        raise KeyError(f"{trial.trial_id}: missing 'aligned_60hz' DataFrame. Build it first.")
    df = trial.dfs["aligned_60hz"]

    # guard: need syllable + both signals
    missing = [c for c in ["syllable", *signal_cols] if c not in df.columns]
    if missing:
        raise KeyError(f"{trial.trial_id}: aligned_60hz missing columns: {missing}")

    onsets = get_onset_positions(df["syllable"])
    traces_by_syl = {}  # {syl_id: {col: [trace,...]}}

    for pos in onsets:
        start = pos - WINDOW_SAMPLES
        end   = pos + WINDOW_SAMPLES
        if start < 0 or end > len(df):
            continue
        syl = df["syllable"].iloc[pos]
        if np.isnan(syl):
            continue
        syl = int(syl)

        bucket = traces_by_syl.setdefault(syl, {c: [] for c in signal_cols})
        window = df.iloc[start:end]

        for col in signal_cols:
            trace = window[col].to_numpy()
            if trace.shape[0] != TOTAL_SAMPLES:
                continue
            sm = _interp_then_smooth(trace, SMOOTHING_WINDOW, SMOOTHING_POLY_ORDER)
            baseline = sm[WINDOW_SAMPLES]  # value at onset
            bucket[col].append(sm - baseline)

    return traces_by_syl

# --- collect per-trial ---
# assume `sessions` is your list of Trial objects
traces_all = []  # list of per-trial dicts
signal_cols = pick_signal_cols(sessions[0].dfs["aligned_60hz"])
print("Using signal columns:", signal_cols)

for t in sessions:
    try:
        traces_all.append(process_trial(t, signal_cols))
    except Exception as e:
        print(f"⚠️ Skipping {getattr(t,'trial_id','?')}: {e}")

# --- count for filtering ---
syll_counts = {}
for d in traces_all:
    for syl, per_col in d.items():
        syll_counts[syl] = syll_counts.get(syl, 0) + sum(len(v) for v in per_col.values())

syllables_to_plot = sorted([s for s, c in syll_counts.items() if c >= MIN_TRACES_PER_SYLLABLE])
print(f"Syllables to plot (min {MIN_TRACES_PER_SYLLABLE} traces):", syllables_to_plot)

time_vector = np.linspace(-WINDOW_SECONDS, WINDOW_SECONDS, TOTAL_SAMPLES)
colors = {"VS": "#1f77b4", "DS": "#2ca02c"}  # blue/green; tweak if you like

# optional genotype split if available
def get_group(trial):
    return getattr(trial, "genotype", "ALL")

# --- plot each syllable: one row, 2 columns (VS, DS); optionally split by genotype ---
for syl in syllables_to_plot:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
    fig.suptitle(f"Syllable Onset: ID {int(syl)}", fontsize=16)

    for j, col in enumerate(signal_cols):
        ax = axes[j]
        # gather traces per group
        grouped = {}
        for trial, per_trial in zip(sessions, traces_all):
            g = get_group(trial)
            traces = per_trial.get(syl, {}).get(col, [])
            if traces:
                grouped.setdefault(g, []).extend(traces)

        # plot each group
        for g, traces in grouped.items():
            valid = [t for t in traces if np.isfinite(t).all()]
            if not valid:
                continue
            arr = np.stack(valid)
            m = arr.mean(axis=0)
            e = sem(arr, axis=0)
            lower, upper = m - CONFIDENCE_LEVEL * e, m + CONFIDENCE_LEVEL * e
            c = colors["VS"] if "VS" in col else colors["DS"]
            label = f"{col} — {g} (n={len(valid)})" if len(grouped) > 1 else f"{col} (n={len(valid)})"
            ax.plot(time_vector, m, label=label)
            ax.fill_between(time_vector, lower, upper, alpha=0.2)

        ax.axvline(0, color="crimson", lw=2, ls="--")
        ax.axhline(0, color="grey",   lw=1, ls=":")
        ax.set_title(col, fontsize=12)
        ax.set_xlabel("Time from onset (s)")
        if j == 0:
            ax.set_ylabel("Δ (baseline @ onset)")
        ax.grid(True, ls=":", alpha=0.5)
        ax.legend(fontsize="small")

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    out_path = os.path.join(fig_base_dir, f"{int(syl)}.png")
    plt.savefig(out_path, dpi=150)
    plt.close(fig)
    print(f"💾 Saved: {out_path}")


In [ ]:
config['bodyparts'] = config['use_bodyparts']
kpms.generate_trajectory_plots(coordinates_est, results, PROJECT_DIR, MODEL_NAME, fps=60, min_frequency=0.000, min_duration=1, **config)

In [ ]:
import os, glob

def map_syllable_parts_to_videos(video_frame_indexes: dict, video_dir: str) -> dict:
    """
    Map KPMS part keys to video files.
    Works with names like:
      part_key: 'T10_Tracking_video2021-06-10T09_27_05_Crop_DLC_resnet50_DLC_IFxDN_SalientMar12shuffle1_760000'
      video:    'T10_Tracking_video2021-06-10T09_27_05_Crop_.mp4'
    """
    video_dir = os.path.expanduser(video_dir)
    video_path_map = {}

    for part_key in video_frame_indexes.keys():
        # Strip any '_partX' suffix KPMS may add
        base = part_key.split('_part')[0]

        # Everything before the first '_DLC_' (preferred) or 'DLC_' (fallback) is the video root
        if '_DLC_' in base:
            root = base.split('_DLC_', 1)[0]
        else:
            root = base.split('DLC_', 1)[0]

        # 1) Direct construct (matches your tree exactly, since root ends with '_Crop_')
        direct = os.path.join(video_dir, root + ".mp4")
        if os.path.isfile(direct):
            video_path_map[part_key] = direct
            continue

        # 2) Fallback: prefix glob (handles slight renames)
        #   e.g. T10_Tracking_video2021-06-10T09_27_05_Crop_*.mp4
        candidates = sorted(glob.glob(os.path.join(video_dir, root + "*.mp4")))
        if candidates:
            video_path_map[part_key] = candidates[0]
        else:
            print(f"Warning: No matching video for key: {part_key}\n  tried: {direct}\n  glob : {root}*.mp4")

    return video_path_map


In [ ]:
# 1) Map h5-part keys to the single source video (SOEREN tree)
VIDEO_DIRECTORY = "/home/fsp585/Desktop/FiberphotometryCode/src_oft/soeren/video_files"

h5_to_video_map_corrected = map_syllable_parts_to_videos(
    video_frame_indexes=video_frame_indexes,
    video_dir=VIDEO_DIRECTORY
)

# 2) (optional) re-extract results if needed
# kpms.reindex_syllables_in_checkpoint(PROJECT_DIR_FULL, MODEL_NAME)
# model, _, _, _ = kpms.load_checkpoint(PROJECT_DIR_FULL, MODEL_NAME)
# results = kpms.extract_results(model, final_metadata, PROJECT_DIR_FULL, MODEL_NAME, save_results=False)

# 3) Align all dicts to the SAME key set
print("--- Aligning input dictionaries to common keys ---")
keys_results = set(results.keys())
keys_frames  = set(video_frame_indexes.keys())
keys_videos  = set(h5_to_video_map_corrected.keys())
keys_coords  = set(coordinates_est.keys())

common_keys = keys_results & keys_frames & keys_videos & keys_coords
print(f"results: {len(keys_results)} | frames: {len(keys_frames)} | videos: {len(keys_videos)} | coords: {len(keys_coords)}")
print(f"→ common: {len(common_keys)}")

# Filter to common keys
results_common             = {k: results[k] for k in common_keys}
video_frame_indexes_common = {k: np.asarray(video_frame_indexes[k], dtype=int) for k in common_keys}
h5_to_video_map_common     = {k: h5_to_video_map_corrected[k] for k in common_keys}
coordinates_common         = {k: coordinates_est[k] for k in common_keys}

# 4) Derive centroids/headings from the SAME coords dict
centroids_common, headings_common = kpms.get_centroids_headings(coordinates_common, **config)

# 5) Avoid fps duplication in config; pass fps explicitly
config_for_movies = config.copy()
config_for_movies.pop('fps', None)

# 6) Generate movies using the SAME project dir you trained with
print("\n--- Generating grid movies ---")
kpms.generate_grid_movies(
    results_common,
    PROJECT_DIR_FULL,           # use your training project
    MODEL_NAME,
    video_paths=h5_to_video_map_common,
    coordinates=coordinates_common,
    centroids=centroids_common,
    headings=headings_common,
    fps=60,
    video_frame_indexes=video_frame_indexes_common,
    overlay_keypoints=False,
    **config_for_movies
)


## Creation of "mother" dataframe

In [ ]:
list(moseq_df.name.unique())

In [ ]:
sessions[1].kpms_h5_path

In [ ]:
sessions[1].dfs.data['synchronized_data_tracking_0']

In [ ]:
moseq_df

In [ ]:
# --- helpers -----------------------------------------
def _base_from_key(k: str) -> str:
    return k.split('_part')[0] if '_part' in k else k

def _part_num_or_none(k: str):
    if '_part' not in k:
        return None
    return int(k.split('_part')[-1])

def _base_from_h5_path(h5_path: str) -> str:
    fname = h5_path.rsplit('/', 1)[-1]
    return fname.rsplit('.', 1)[0]

# --- 1) Build per-session USED row arrays (concat parts in order) ------------
base_to_used_rows = {}
for k, arr in video_frame_indexes.items():
    b = _base_from_key(k)
    base_to_used_rows.setdefault(b, []).append((k, _part_num_or_none(k), arr))

for b, items in base_to_used_rows.items():
    if len(items) == 1 and items[0][1] is None:
        base_to_used_rows[b] = items[0][2]
    else:
        items_sorted = sorted(items, key=lambda t: t[1])
        base_to_used_rows[b] = np.concatenate([t[2] for t in items_sorted])

video_bases = set(base_to_used_rows.keys())

# --- 2) Tag MoSeq with session_base (from the filename embedded in `name`) ---
import re
pattern = "(" + "|".join(re.escape(b) for b in sorted(video_bases)) + ")"
moseq_tagged = moseq_df.copy()
moseq_tagged["session_base"] = moseq_tagged["name"].str.extract(pattern, expand=False)

# Optional: quick flag if a MoSeq row’s frame_index would be out of bounds for its session
def _flag_in_video_index(group):
    used = base_to_used_rows.get(group.name, None)
    if used is None or len(used) == 0:
        group["in_video_index"] = False
    else:
        n = len(used)
        fi = group["frame_index"].to_numpy()
        group["in_video_index"] = (fi >= 0) & (fi < n)
    return group

moseq_tagged = (
    moseq_tagged
    .dropna(subset=["session_base"])
    .groupby("session_base", group_keys=False)
    .apply(_flag_in_video_index)
)

# --- 3) Build a single, annotated signal DF with a MoSeq-like index ----------
# For each session:
#   - get its sync DF
#   - compute inverse map: current_h5_row -> local MoSeq frame_index (position in used array)
#   - add `session_base` and `moseq_frame_index` columns
sync_chunks = []
for sess in sessions:
    base = _base_from_h5_path(sess.kpms_h5_path)
    sync_df = sess.dfs.data.get("synchronized_data_tracking_0", None)
    if (base not in base_to_used_rows) or (sync_df is None):
        continue

    used = base_to_used_rows[base]  # array of current_h5_row values used by MoSeq
    # inverse map: for each used current_h5_row, what is its local MoSeq index?
    inv = pd.Series(np.arange(len(used), dtype=np.int64), index=used)

    s = sync_df.copy()
    if "current_h5_row" not in s.columns:
        s = s.reset_index().rename(columns={"index": "current_h5_row"})

    s["session_base"] = base
    s["moseq_frame_index"] = s["current_h5_row"].map(inv).astype("Int64")  # NaN where signal row wasn’t used by MoSeq
    sync_chunks.append(s)

sync_all = pd.concat(sync_chunks, ignore_index=True) if sync_chunks else pd.DataFrame()

# --- 4) Merge: MoSeq (left) x annotated signal on (session_base, frame_index) --
# Guarantees len(output) == len(moseq_df); unmatched signal rows are not included.
merged_all = moseq_tagged.merge(
    sync_all,
    left_on=["session_base", "frame_index"],
    right_on=["session_base", "moseq_frame_index"],
    how="left",
    suffixes=("", "_fp"),
    indicator=True
)
merged_all["had_signal_match"] = merged_all["_merge"].eq("both")
merged_all.drop(columns=["_merge"], inplace=True)

# --- 5) (optional) tidy columns ------------------------------------------------
preferred_left = [
    "session_base", "name", "group", "syllable", "onset",
    "frame_index", "in_video_index", "had_signal_match"
]
left_cols = [c for c in preferred_left if c in merged_all.columns]
other_cols = [c for c in merged_all.columns if c not in left_cols]
merged_all = merged_all[left_cols + other_cols]

# Optional stable sort (per session, by MoSeq order)
# merged_all.sort_values(["session_base", "frame_index"], inplace=True, kind="stable")


In [ ]:
moseq_df

In [ ]:
merged_all.columns

## Syllable entropy

Does the KI have less behavioral diversity and longer bouts?

## Transition entropy

Does the KI show less sequence entropy?

## Event-aligned DA at onset

How does the DS/VS act during exploratory vs repetitive syllables across genotypes?

## Kinematics of syllables

How does the execution of syllables differ across genotypes? Is there a coupling between the DA of a syllable and the ensuing vigor of said syllable instance?

## Can kinematics and syllable signature predict said syllable?

## Heatmap of syllable execution

What syllables are (not) center biased and which are more neutral? Do syllable signatures change depending on location and genotype?

## Linear mixed models

Using mixed models, can we get a more clear indication of significant differences in genotype response?

##